# PNAD — deterministic upper-tail trimming

This notebook reads the annual refined PNAD datasets produced by `01_gera_datasets.ipynb` and creates the corresponding trusted datasets in `dados_trusted`. The refined files already contain the upstream deterministic cleaning step: invalid income records and metadata-defined missing-income sentinels have been removed. Therefore, this notebook performs only the **annual statistical upper-tail trim**. For each year,

$$
z_i=\log(1+x_i),
$$

and the robust annual center and dispersion are

$$
m=\operatorname{median}(z_i),
\qquad
s=1.4826\,\operatorname{median}\left(|z_i-m|\right).
$$

The deterministic upper cutoff is

$$
x_c=\exp(m+k s)-1,
$$

with \(k=6\) by default. Only observations with \(x_i>x_c\) are removed. No lower-tail trimming is applied. The implementation deliberately avoids unnecessary DataFrame conversions, repeated quantile calculations, index reconstruction, and duplicate copies of the annual income vector. The statistical rule is unchanged; only the computational implementation is simplified. Outputs:

- `dados_trusted/pnad_trusted_YYYY.parquet`;
- `assets/data/outlier_trim_audit.csv`.


# Functions

In [1]:
from pathlib import Path
from time import perf_counter
import re

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", None)

PROJECT_PATH = Path(
    r"C:\Users\Osvaldo\OneDrive\academic_research\econophysics\projeto_pnad_ic_beatriz"
)

REFINED_PATH = PROJECT_PATH / "dados_refined"
TRUSTED_PATH = PROJECT_PATH / "dados_trusted"
DATA_ASSETS_PATH = PROJECT_PATH / "assets" / "data"

REFINED_PATTERN = "pnad_refined_*.parquet"

MAD_CONSISTENCY = 1.4826
MAD_THRESHOLD = 6.0
PARQUET_ENGINE = "pyarrow"
PARQUET_COMPRESSION = "snappy"

if not REFINED_PATH.is_dir():
    raise FileNotFoundError(f"Refined-data directory not found: {REFINED_PATH}")

TRUSTED_PATH.mkdir(parents=True, exist_ok=True)
DATA_ASSETS_PATH.mkdir(parents=True, exist_ok=True)


def year_from_filename(path):
    """Extract the four-digit year from a refined PNAD filename."""
    match = re.fullmatch(r"pnad_refined_(\d{4})", path.stem)
    if not match:
        raise ValueError(f"Invalid refined filename: {path.name}")
    return int(match.group(1))


def discover_refined_files(refined_path=REFINED_PATH):
    """Return the refined annual Parquet files ordered by year."""
    files = {
        year_from_filename(path): path
        for path in refined_path.glob(REFINED_PATTERN)
    }

    if not files:
        raise FileNotFoundError(
            f"No files matching '{REFINED_PATTERN}' were found in {refined_path}."
        )

    return dict(sorted(files.items()))


def validate_refined_frame(df, year):
    """Validate only the assumptions needed by the trimming rule."""
    required = {"renda", "ano"}
    missing = required.difference(df.columns)

    if missing:
        raise ValueError(
            f"{year}: missing required columns: {', '.join(sorted(missing))}"
        )

    if not df["ano"].eq(year).all():
        raise ValueError(f"{year}: inconsistent values in column 'ano'.")

    income = df["renda"].to_numpy(dtype=np.float64, copy=False)

    if income.size == 0:
        raise ValueError(f"{year}: empty refined dataset.")

    if not np.isfinite(income).all():
        raise ValueError(f"{year}: non-finite refined income values found.")

    if (income < 0).any():
        raise ValueError(f"{year}: negative refined income values found.")

    return income


def compute_log_mad_threshold(
    income,
    threshold=MAD_THRESHOLD,
    consistency=MAD_CONSISTENCY,
):
    """
    Compute the deterministic annual upper cutoff.

    Only one transformed working array is allocated. After the transformed
    median is computed, that same array is reused to compute the MAD.
    """
    transformed = np.log1p(income)

    center = float(np.median(transformed))

    transformed -= center
    np.abs(transformed, out=transformed)

    mad = float(np.median(transformed))
    dispersion = float(consistency * mad)
    dispersion_method = "scaled_mad"

    if dispersion <= 0 and income.size > 1:
        transformed = np.log1p(income)
        dispersion = float(np.std(transformed, ddof=1))
        dispersion_method = "std_fallback"

    if dispersion > 0:
        cutoff = float(
            np.expm1(center + float(threshold) * dispersion)
        )
    else:
        cutoff = float(np.max(income))
        dispersion_method = "zero_dispersion"

    return {
        "median_log1p": center,
        "mad_log1p": mad,
        "scaled_dispersion_log1p": dispersion,
        "dispersion_method": dispersion_method,
        "mad_consistency": float(consistency),
        "threshold_k": float(threshold),
        "statistical_cutoff": cutoff,
    }


def trim_refined_year(
    df,
    year,
    threshold=MAD_THRESHOLD,
    consistency=MAD_CONSISTENCY,
):
    """
    Return the trusted annual frame and the deterministic trim audit.

    The trusted frame preserves the original columns and dtypes. The original
    index is irrelevant because the Parquet file is written with index=False.
    """
    income = validate_refined_frame(df, year)

    threshold_info = compute_log_mad_threshold(
        income,
        threshold=threshold,
        consistency=consistency,
    )

    cutoff = threshold_info["statistical_cutoff"]
    keep = income <= cutoff

    n_refined = int(income.size)
    n_trusted = int(np.count_nonzero(keep))
    n_removed = n_refined - n_trusted

    trusted = df.loc[keep]

    audit = {
        "year": int(year),
        "n_refined": n_refined,
        **threshold_info,
        "maximum_before": float(np.max(income)),
        "n_statistical_outlier": n_removed,
        "removal_rate": n_removed / n_refined,
        "n_trusted": n_trusted,
        "maximum_after": (
            float(np.max(income[keep]))
            if n_trusted > 0
            else np.nan
        ),
    }

    return trusted, audit

## Functions - trusted datasets

In [2]:
def build_trusted_datasets(
    refined_path=REFINED_PATH,
    trusted_path=TRUSTED_PATH,
    audit_path=DATA_ASSETS_PATH / "df_outlier_trim_audit.csv",
    threshold=MAD_THRESHOLD,
    consistency=MAD_CONSISTENCY,
    all_columns=True,
):
    """
    Build annual trusted PNAD datasets from the refined Parquet files.

    Each annual dataset is read, trimmed using the deterministic log-MAD
    upper-tail rule, and written to the trusted-data directory.

    The annual trimming audit is automatically saved as a CSV file.

    Parameters
    ----------
    all_columns : bool, default=True
        If True, return the complete audit table. If False, return only
        the main analytical and timing columns.

    Returns
    -------
    pd.DataFrame
        Annual outlier-trimming audit.
    """
    files_by_year = discover_refined_files(refined_path)
    audit_rows = []

    progress = tqdm(
        files_by_year.items(),
        total=len(files_by_year),
        desc="Building trusted PNAD datasets",
        unit="year",
    )

    for year, input_path in progress:
        progress.set_postfix_str(f"{year}: reading")

        t0 = perf_counter()

        df_refined = pd.read_parquet(
            input_path,
            engine=PARQUET_ENGINE,
        )

        t1 = perf_counter()

        progress.set_postfix_str(f"{year}: trimming")

        df_trusted, audit = trim_refined_year(
            df_refined,
            year,
            threshold=threshold,
            consistency=consistency,
        )

        t2 = perf_counter()

        progress.set_postfix_str(f"{year}: writing")

        output_path = trusted_path / f"pnad_trusted_{year}.parquet"

        df_trusted.to_parquet(
            output_path,
            index=False,
            engine=PARQUET_ENGINE,
            compression=PARQUET_COMPRESSION,
        )

        t3 = perf_counter()

        audit.update({
            "input_file": input_path.name,
            "output_file": output_path.name,
            "read_seconds": t1 - t0,
            "trim_seconds": t2 - t1,
            "write_seconds": t3 - t2,
            "total_seconds": t3 - t0,
        })

        audit_rows.append(audit)

        del df_refined, df_trusted

    df_outlier_trim_audit = (
        pd.DataFrame(audit_rows)
        .sort_values("year")
        .reset_index(drop=True)
    )

    valid_counts = (
        df_outlier_trim_audit["n_refined"]
        == df_outlier_trim_audit["n_trusted"]
        + df_outlier_trim_audit["n_statistical_outlier"]
    )

    if not valid_counts.all():
        raise RuntimeError(
            "Trusted-data row counts failed the trimming audit."
        )

    audit_path.parent.mkdir(parents=True, exist_ok=True)

    df_outlier_trim_audit.to_csv(
        audit_path,
        index=False,
    )

    if all_columns:
        return df_outlier_trim_audit

    columns = [
        "year",
        "n_refined",
        "statistical_cutoff",
        "maximum_before",
        "n_statistical_outlier",
        "removal_rate",
        "n_trusted",
        "maximum_after",
        "read_seconds",
        "trim_seconds",
        "write_seconds",
        "total_seconds",
    ]

    return df_outlier_trim_audit[columns]

# Pipeline: trusted datasets

Each annual refined dataset is read once, trimmed once, and written once. The progress table records separate read, trim, and write times so that computational work can be distinguished from filesystem or OneDrive I/O. The trusted Parquet files preserve the same two-column structure generated by the refined-data pipeline: `renda` and `ano`. The audit stores the exact annual cutoff, the number and fraction of removed observations, the upper observed values before and after trimming, and the elapsed time for each processing phase. The timing columns are operational diagnostics only; they do not enter the scientific analysis.


In [3]:
df_outlier_trim_audit = build_trusted_datasets(
                                                threshold=6.0,
                                                consistency=1.4826,
                                                all_columns=False,
                                                #columns=[],
                                              )

df_outlier_trim_audit

Building trusted PNAD datasets:   0%|          | 0/45 [00:00<?, ?year/s]

,year,n_refined,statistical_cutoff,maximum_before,n_statistical_outlier,removal_rate,n_trusted,maximum_after,read_seconds,trim_seconds,write_seconds,total_seconds
0,1976,148396,4.725177e+05,9.999990e+05,9,0.000061,148387,4.055440e+05,0.049646,0.008518,0.009040,0.067204
1,1977,193886,6.945145e+05,1.119000e+06,927,0.004781,192959,6.925000e+05,0.004463,0.009296,0.010618,0.024376
2,1978,216329,9.508754e+05,1.099998e+06,634,0.002931,215695,8.300000e+05,0.004094,0.009670,0.013734,0.027497
3,1979,171371,1.425020e+06,1.029998e+06,0,0.000000,171371,1.029998e+06,0.004253,0.007124,0.009924,0.021301
4,1981,472198,2.379649e+06,8.300000e+05,0,0.000000,472198,8.300000e+05,0.007198,0.013332,0.025634,0.046164
5,1982,206845,1.190481e+07,5.150000e+06,0,0.000000,206845,5.150000e+06,0.003276,0.008334,0.013546,0.025156
6,1983,499797,1.174398e+07,7.000000e+06,0,0.000000,499797,7.000000e+06,0.007437,0.014849,0.038046,0.060332
7,1984,504543,2.857385e+07,9.999998e+06,0,0.000000,504543,9.999998e+06,0.007249,0.013728,0.029200,0.050177
8,1985,514137,1.337692e+08,7.500000e+07,0,0.000000,514137,7.500000e+07,0.006711,0.014777,0.029910,0.051398
9,1986,284142,3.435870e+05,3.346667e+05,0,0.000000,284142,3.346667e+05,0.004826,0.008319,0.016353,0.029498
